In [6]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("❌ GPU is NOT available")

CUDA available: True
GPU: Tesla T4
GPU count: 2


In [7]:
from pathlib import Path

input_root = Path("/kaggle/input")
gguf_files = sorted(input_root.rglob("*.gguf"))

print("Input root exists:", input_root.exists())
print("GGUF shards found:", len(gguf_files))

for path in gguf_files:
    print(path)
    print(f"  size: {path.stat().st_size:,} bytes")
    print(f"  size: {path.stat().st_size / 1024**3:.3f} GiB")

expected_names = {
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
}
found_names = {path.name for path in gguf_files}
missing = expected_names - found_names

if missing:
    raise FileNotFoundError(f"Missing expected model shards: {sorted(missing)}")

model_path = next(path for path in gguf_files if "00001-of-00002" in path.name)
total_size = sum(path.stat().st_size for path in gguf_files)

print(f"Model path: {model_path}")
print(f"Total GGUF size: {total_size / 1024**3:.3f} GiB")
print("Model file access: OK")

Input root exists: True
GGUF shards found: 2
/kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
  size: 3,993,201,344 bytes
  size: 3.719 GiB
/kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local/qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf
  size: 689,872,288 bytes
  size: 0.642 GiB
Model path: /kaggle/input/datasets/keshav0210/qwen25-7b-instruct-q4-km-local/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
Total GGUF size: 4.361 GiB
Model file access: OK


In [8]:
import os
from pathlib import Path

print("Remote Python:", os.sys.executable)
print("Remote working directory:", Path.cwd())
print("Project source present:", (Path.cwd() / "src" / "mastercard_defence").exists())
print("Working directory entries:")
for path in sorted(Path.cwd().iterdir()):
    print(" ", path)

Remote Python: /usr/bin/python3
Remote working directory: /kaggle/working
Project source present: False
Working directory entries:
  /kaggle/working/.virtual_documents


In [27]:
import subprocess
import sys
from pathlib import Path

project_dir = Path("/kaggle/working/mastercard_hackathon")
repository_url = "https://github.com/keshav-0210/mastercard_hackathon.git"

if not project_dir.exists():
    subprocess.run(
        ["git", "clone", repository_url, str(project_dir)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)

source_root = project_dir / "src"
sys.path.insert(0, str(source_root))

required_paths = [
    source_root / "mastercard_defence",
    project_dir / "config" / "default.yaml",
    project_dir / "data" / "knowledge_base",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing synced project paths: {missing}")

print("Project synced:", project_dir)
print("Source import path:", source_root)
print("Project sync: OK")

Updating 8541eaa..7ebbac5
Fast-forward
 src/mastercard_defence/agents.py | 11 +++++++++++
 1 file changed, 11 insertions(+)
Project synced: /kaggle/working/mastercard_hackathon
Source import path: /kaggle/working/mastercard_hackathon/src
Project sync: OK


From https://github.com/keshav-0210/mastercard_hackathon
   8541eaa..7ebbac5  main       -> origin/main


In [10]:
import os

os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)

from mastercard_defence.loop import load_config
from mastercard_defence.runtime import require_model

config_path = project_dir / "config" / "default.yaml"
config = load_config(str(config_path))
validated_model_path = require_model(config)

try:
    import llama_cpp
    print("llama_cpp version:", getattr(llama_cpp, "__version__", "unknown"))
    print("Project imports: OK")
    print("Model configuration: OK")
except ImportError:
    print("llama_cpp is not installed in the Kaggle kernel")

llama_cpp is not installed in the Kaggle kernel


In [11]:
%env CMAKE_ARGS=-DGGML_CUDA=on
%pip install --no-cache-dir llama-cpp-python

env: CMAKE_ARGS=-DGGML_CUDA=on
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 244.5 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 213.6 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
Failed to build llama-cpp-python
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (llama-cpp-python)
Note: you may need to restart the kernel to use updated packages.


In [12]:
import subprocess
import torch

print("Torch CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("NVIDIA runtime:")
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], check=False)

Torch CUDA version: 12.8
CUDA available: True
GPU: Tesla T4
NVIDIA runtime:
Tesla T4, 580.159.04
Tesla T4, 580.159.04


CompletedProcess(args=['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], returncode=0)

In [13]:
%pip install --no-cache-dir --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu128 llama-cpp-python

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 312.5 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 241.4 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
Failed to build llama-cpp-python
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (llama-cpp-python)
Note: you may need to restart the kernel to use updated packages.


In [14]:
%pip install --no-cache-dir --index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --extra-index-url https://pypi.org/simple llama-cpp-python

Looking in indexes: https://abetlen.github.io/llama-cpp-python/whl/cu124, https://pypi.org/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 237.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 28.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [15]:
import json
import time

from llama_cpp import llama_supports_gpu_offload
from mastercard_defence.llm import SharedLocalLLM

print("llama.cpp GPU offload support:", llama_supports_gpu_offload())
if not llama_supports_gpu_offload():
    raise RuntimeError("Installed llama.cpp does not support GPU offload")

llm = SharedLocalLLM(config)
start = time.perf_counter()
response = llm.complete_json(
    system_prompt="You are a payment-security research assistant. Return only valid JSON.",
    user_prompt=(
        "Create one synthetic payment-security attack hypothesis. "
        "Return exactly the keys attack_family, scenario, and research_direction."
    ),
)
elapsed = time.perf_counter() - start

print("Model JSON response:")
print(json.dumps(response, indent=2))
print(f"Inference time: {elapsed:.2f} seconds")
print("Qwen GPU inference smoke test: OK")

ggml_cuda_init: found 2 CUDA devices (Total VRAM: 29823 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB
  Device 1: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14911 MiB


llama.cpp GPU offload support: True


TypeError: Llama.__call__() got an unexpected keyword argument 'messages'

In [16]:
import json
import time

start = time.perf_counter()
response = llm._load().create_chat_completion(
    messages=[
        {
            "role": "system",
            "content": "You are a payment-security research assistant. Return only valid JSON.",
        },
        {
            "role": "user",
            "content": (
                "Create one synthetic payment-security attack hypothesis. "
                "Return exactly the keys attack_family, scenario, and research_direction."
            ),
        },
    ],
    response_format={"type": "json_object"},
    temperature=0.2,
    max_tokens=300,
)
content = response["choices"][0]["message"]["content"]
parsed_response = json.loads(content)
print(json.dumps(parsed_response, indent=2))
print(f"Inference time: {time.perf_counter() - start:.2f} seconds")
print("Qwen GPU inference smoke test: OK")

{
  "attack_family": "Man-in-the-Middle",
  "scenario": "An attacker intercepts the communication between a mobile app and the payment gateway by exploiting a weak SSL/TLS implementation, allowing them to read and modify payment data.",
  "research_direction": "Developing more robust SSL/TLS configurations and implementing additional security measures such as certificate pinning to prevent such attacks."
}
Inference time: 6.63 seconds
Qwen GPU inference smoke test: OK


In [28]:
import importlib
import os
import subprocess
import sys

subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)
os.chdir(project_dir)

for module_name in list(sys.modules):
    if module_name == "mastercard_defence" or module_name.startswith("mastercard_defence."):
        del sys.modules[module_name]

sys.path.insert(0, str(project_dir / "src"))
from mastercard_defence.loop import ClosedLoop, load_config

os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)
config = load_config("config/default.yaml")
loop = ClosedLoop(config)
try:
    results = loop.run(rounds=2)
finally:
    loop.close()

assert len(results) == 2
assert all(result["hypothesis"].evidence for result in results)
assert results[0]["weakness"].round_id == 1
assert results[1]["weakness"].round_id == 2

print("Rounds completed:", len(results))
print("Agent backend:", type(loop.agents).__name__)
print("Attack families:", [result["hypothesis"].attack_family for result in results])
print("Detection F1:", [round(result["detection"]["f1"], 3) for result in results])
print("Agent 3 -> Memory -> Agent 1 loop: OK")

Already up to date.


: 

: 

: 

In [ ]:
from mastercard_defence.agents import QwenAgents
from mastercard_defence.llm import SharedLocalLLM
from mastercard_defence.loop import ClosedLoop, load_config

os.environ["RUN_MODE"] = "KAGGLE_GPU"
os.environ["MODEL_PATH"] = str(model_path)
config = load_config("config/default.yaml")
shared_llm = SharedLocalLLM(config)
qwen_agents = QwenAgents(config, llm=shared_llm)
loop = ClosedLoop(config, agents=qwen_agents)
try:
    results = loop.run(rounds=2)
finally:
    loop.close()

assert len(results) == 2
assert all(result["hypothesis"].evidence for result in results)
assert results[0]["weakness"].round_id == 1
assert results[1]["weakness"].round_id == 2

print("Rounds completed:", len(results))
print("Agent backend:", type(loop.agents).__name__)
print("Attack families:", [result["hypothesis"].attack_family for result in results])
print("Detection F1:", [round(result["detection"]["f1"], 3) for result in results])
print("Agent 3 -> Memory -> Agent 1 loop: OK")

In [18]:
import subprocess

print("Git revision:")
subprocess.run(["git", "-C", str(project_dir), "log", "-1", "--oneline"], check=True)
agent_source = (project_dir / "src" / "mastercard_defence" / "agents.py").read_text(encoding="utf-8")
print("QwenAgents present in Kaggle source:", "class QwenAgents" in agent_source)
print("QwenAgents selected in current process:", type(loop.agents).__name__)

Git revision:
4f25c38 Add first-cut AI Defence Lab pipeline
QwenAgents present in Kaggle source: False
QwenAgents selected in current process: HeuristicAgents
